In [8]:
!pip uninstall -y langchain langchain-openai langchain-community chromadb
!pip install -q langchain langchain-openai langchain-community chromadb pypdf rank_bm25

Found existing installation: langchain 1.2.10
Uninstalling langchain-1.2.10:
  Successfully uninstalled langchain-1.2.10
Found existing installation: langchain-openai 1.1.10
Uninstalling langchain-openai-1.1.10:
  Successfully uninstalled langchain-openai-1.1.10
Found existing installation: langchain-community 0.4.1
Uninstalling langchain-community-0.4.1:
  Successfully uninstalled langchain-community-0.4.1
Found existing installation: chromadb 1.5.1
Uninstalling chromadb-1.5.1:
  Successfully uninstalled chromadb-1.5.1


You can safely remove it manually.


### **Hybrid Search in RAG**
(Vector Search + Keyword Search using LangChain + ChromaDB)

Hybrid search combines:

Semantic similarity (vector embeddings)

Keyword-based search (BM25 / lexical matching)

This improves retrieval quality because:

Vector search captures meaning

Keyword search captures exact terms

Together → higher recall + better precision

In [ ]:
import os
from openai import OpenAI

os.environ["OPENAI_API_KEY"] = 

client = OpenAI()

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "user", "content": "Give me a short definition of DNN."}
    ]
)

print(response.choices[0].message.content)

A Deep Neural Network (DNN) is a type of artificial neural network with multiple layers of interconnected nodes, or neurons, that processes data to learn complex patterns and representations. DNNs are widely used in various applications such as image and speech recognition, natural language processing, and more due to their ability to model intricate relationships in large datasets.


In [10]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import Chroma
from langchain_community.retrievers import BM25Retriever
from langchain_openai import OpenAIEmbeddings, ChatOpenAI

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

In [11]:
# from google.colab import files
# uploaded = files.upload()

#pdf_path = list(uploaded.keys())[0]
pdf_path = "C:\\Users\\Administrator\\Documents\\GenAI Training\\Milestone\\Senior Engineering Checklist.pdf"
loader = PyPDFLoader(pdf_path)
documents = loader.load()

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=200
)

chunks = text_splitter.split_documents(documents)

In [12]:
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./chroma_db"
)

vector_retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

In [13]:
bm25_retriever = BM25Retriever.from_documents(chunks)
bm25_retriever.k = 4

In [14]:
def hybrid_retrieval(query):
    vector_docs = vector_retriever.invoke(query)
    keyword_docs = bm25_retriever.invoke(query)

    # Merge results
    combined = vector_docs + keyword_docs

    # Remove duplicates
    unique_docs = list({doc.page_content: doc for doc in combined}.values())

    return unique_docs

In [15]:
prompt = ChatPromptTemplate.from_template("""
You are an expert assistant.

Use ONLY the context below to answer.
If not found, say you don't know.

Context:
{context}

Question:
{question}

Answer:
""")

In [16]:
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

In [17]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

hybrid_chain = (
    {
        "context": lambda x: format_docs(hybrid_retrieval(x)),
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

In [18]:
query = "Explain the key points."

response = hybrid_chain.invoke(query)

print(response)

The key points from the context are as follows:

1. **Checklist for Project Structure**: 
   - Maintain a clear folder structure including sections for Requirements, Architecture & Design, Data Flow, Implementation/Code, Testing, and Readme.

2. **Distinction Between Data Flow and Architecture**: 
   - Data flow (e.g., ingestion and retrieval) should not be confused with architecture.

3. **Architecture Highlights**: 
   - Architecture should emphasize different software components, microservices, and caching mechanisms.

4. **Observability**: 
   - Implement structured logging (JSON) and health check endpoints (/health, /ready) for monitoring.

5. **Advanced User Experience**: 
   - Use techniques like Server-Sent Events (SSE) for better performance in long-running tasks.

6. **Documentation and Decision Logic**: 
   - Document architectural decisions (Architecture Decision Records) and the reasoning behind technology choices, including trade-offs.

7. **Production-Grade Code**: 
   -